# Phase 2 & 3: Agro-Climatic Reference Calendar & Lag-Aware Machine Learning Specification

**Project:** Rainfall-Lag-Aware Crop Yield Forecasting in India Using Explainable Temporal Machine Learning

**Objective:**
Establish the official phenological reference calendar for India's 15 Agro-Climatic Planning Zones, formulate the 16-week pre-harvest rainfall lag sequence ($w_{-16}$ to $w_{-1}$), and specify the lag-aware tabular (Lag-RF, Lag-XGBoost) and recurrent sequence (LSTM, GRU) architectures.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch

sys.path.insert(0, str(Path.cwd()))

from src.crop_calendar import build_crop_calendar_dataframe, get_crop_reference_week, get_pre_harvest_lag_weeks
from src.lag_construction import extract_16week_lag_sequence, LAG_COLUMN_NAMES
from src.lag_models import LagAwareTabularPipeline, extract_lag_importance_profile
from src.sequence_models import CropYieldSequenceDataset, RiceLSTM, RiceGRU

print(f'PyTorch Version: {torch.__version__}')
print('Phase 2 & 3 modules loaded successfully.')

PyTorch Version: 2.14.0+cpu
Phase 2 & 3 modules loaded successfully.


## 1. Agro-Climatic Reference Periods
Documenting phenological windows across India's 15 Agro-Climatic Zones based on Ministry of Agriculture / ICAR standards.

In [2]:
cal_df = build_crop_calendar_dataframe()
cal_df[['Zone_ID', 'Agro_Climatic_Zone', 'Representative_States', 'Harvest_Reference_Week', 'Harvest_Month', '16_Week_Lag_Window']]

,Zone_ID,Agro_Climatic_Zone,Representative_States,Harvest_Reference_Week,Harvest_Month,16_Week_Lag_Window
0,1,Western Himalayan Region,"Himachal Pradesh, Uttarakhand",41,October,Weeks 25 to 40
1,2,Eastern Himalayan Region,Assam,46,November,Weeks 30 to 45
2,3,Lower Gangetic Plain Region,West Bengal,46,November,Weeks 30 to 45
3,4,Middle Gangetic Plain Region,Bihar,45,November,Weeks 29 to 44
4,5,Upper Gangetic Plain Region,Uttar Pradesh,43,October,Weeks 27 to 42
5,6,Trans-Gangetic Plain Region,"Punjab, Haryana",42,October,Weeks 26 to 41
6,7,Eastern Plateau and Hill Region,"Chhattisgarh, Jharkhand",45,November,Weeks 29 to 44
7,8,Central Plateau and Hill Region,"Madhya Pradesh, Rajasthan",43,October,Weeks 27 to 42
8,9,Western Plateau and Hill Region,Maharashtra,44,October,Weeks 28 to 43
9,10,Southern Plateau and Hill Region,"Telangana, Karnataka",46,November,Weeks 30 to 45


## 2. 16-Week Pre-Harvest Lag Sequence Extraction & Causality Verification
Demonstrating the exact causal backward-looking extraction: $w_{-16}$ through $w_{-1}$ where $w_{-1} < T_{\text{ref}}$.

In [3]:
for state in ['Punjab', 'Kerala', 'West Bengal', 'Bihar', 'Tamil Nadu']:
    t_ref = get_crop_reference_week(state)
    lag_wks = get_pre_harvest_lag_weeks(t_ref, lag_window_size=16)
    print(f'{state:15s} | Harvest Week T_ref: {t_ref:2d} | 16-Week Lag Range: Weeks {lag_wks[0]:2d} to {lag_wks[-1]:2d} (Strictly < {t_ref})')

Punjab          | Harvest Week T_ref: 42 | 16-Week Lag Range: Weeks 26 to 41 (Strictly < 42)
Kerala          | Harvest Week T_ref: 41 | 16-Week Lag Range: Weeks 25 to 40 (Strictly < 41)
West Bengal     | Harvest Week T_ref: 46 | 16-Week Lag Range: Weeks 30 to 45 (Strictly < 46)
Bihar           | Harvest Week T_ref: 45 | 16-Week Lag Range: Weeks 29 to 44 (Strictly < 45)
Tamil Nadu      | Harvest Week T_ref: 47 | 16-Week Lag Range: Weeks 31 to 46 (Strictly < 47)


## 3. Deep Sequence Architectures: PyTorch RiceLSTM & RiceGRU
Validating multi-input sequential tensor fusion (16-week rainfall sequence + static spatial fixed effects).

In [4]:
B = 4   # Batch size
L = 16  # Pre-harvest sequence length
F_seq = 1
F_static = 20

dummy_seq = torch.randn(B, L, F_seq)
dummy_static = torch.randn(B, F_static)

lstm_model = RiceLSTM(seq_input_dim=F_seq, static_input_dim=F_static, hidden_dim=32, num_layers=2)
gru_model = RiceGRU(seq_input_dim=F_seq, static_input_dim=F_static, hidden_dim=32, num_layers=2)

lstm_out = lstm_model(dummy_seq, dummy_static)
gru_out = gru_model(dummy_seq, dummy_static)

print(f'LSTM Output Shape: {lstm_out.shape} -> Expected (B={B}, 1)')
print(f'GRU Output Shape:  {gru_out.shape} -> Expected (B={B}, 1)')
print(f'Sample LSTM Predictions: {lstm_out.squeeze().detach().numpy()}')

LSTM Output Shape: torch.Size([4, 1]) -> Expected (B=4, 1)
GRU Output Shape:  torch.Size([4, 1]) -> Expected (B=4, 1)
Sample LSTM Predictions: [0.08379753 0.07184375 0.09735376 0.10817681]
